In [ ]:
import pandas as pd
import os
from pathlib import Path
from thefuzz import process

# 1. Load the processed data
PROCESSED_DIR = Path('../Data/processed')
df_events = pd.read_csv(PROCESSED_DIR / 'events/master_events.csv')
df_clubs = pd.read_csv(PROCESSED_DIR / 'clubs/master_clubs.csv')
df_matches = pd.read_csv(PROCESSED_DIR / 'events/master_match_results.csv')


In [ ]:

def search_clubs(query, match_df=df_clubs):
    """
    Performs a case-insensitive search for clubs matching the query.
    Returns a unique list of club names.
    """
    # 1. Extract unique club names from the data
    unique_clubs = pd.Series(match_df['Name'].unique())
    
    # 2. Filter using case-insensitive partial matching
    matches = unique_clubs[unique_clubs.str.contains(query, case=False, na=False)]
    
    # 3. Present results
    if matches.empty:
        print(f"No clubs found matching: '{query}'")
        return []
    else:
        print(f"Found {len(matches)} potential matches for '{query}':")
        return matches.tolist()

# --- Now you can run the search ---
results = search_clubs("Ocean")

if results:
    # Example: Picking the first result for further analysis
    my_target_club = results[0]
    print(f"\nTarget Club Selected: {my_target_club}")


In [ ]:
# 1. Search to find the exact spelling
results = search_clubs("Ocean")

# 2. Select the correct one (e.g., the first result)
my_target_club = results[0] 

# 3. Run your analysis using that exact name
club_matches = df_clubs[df_clubs['Name'] == my_target_club]



In [ ]:
from thefuzz import process

def fuzzy_find_clubs(query, limit=10, match_df=df_clubs):
    """
    Uses Levenshtein Distance to find the closest matches even with typos.
    """
    unique_names = match_df['Name'].unique().tolist()
    
    # Extract the top 'limit' matches
    results = process.extract(query, unique_names, limit=limit)
    
    # Format: (Name, Confidence Score)
    print(f"Top matches for '{query}':")
    for name, score in results:
        print(f"[{score}% match] - {name}")

# --- Example Usage ---
# Even if you misspell it, it will find the closest name
fuzzy_find_clubs("MVP") 


In [ ]:
def get_teams_in_club(club_name, match_df=df_matches):
    """
    Finds all unique team names in the match data that belong to the specified club.
    """
    # 1. Get all unique names from both Team A and Team B columns
    all_teams = pd.concat([match_df['Team_A_Name'], match_df['Team_B_Name']]).unique()
    all_teams_series = pd.Series(all_teams).dropna()
    
    # 2. Filter teams that contain the club name
    # We use case-insensitive matching
    club_teams = all_teams_series[all_teams_series.str.contains(club_name, case=False)]
    
    if club_teams.empty:
        print(f"No teams found containing the name '{club_name}'.")
        return []
    else:
        print(f"Found {len(club_teams)} teams for club '{club_name}':")
        return sorted(club_teams.tolist())

# --- Example Usage ---
# Use the exact club name found from your 'search_clubs' routine
my_teams = get_teams_in_club("MVP")
print(my_teams)


In [ ]:
import pandas as pd
import os
from pathlib import Path
from thefuzz import process

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================
DATA_DIR = Path('../Data/processed')
WATCHLIST_PATH = DATA_DIR / 'regional_watchlist.csv'

# Define paths for the master data files
RESULTS_PATH = DATA_DIR / 'events/master_match_results.csv'
CLUBS_PATH = DATA_DIR / 'clubs/master_clubs.csv'

# =============================================================================
# DATA PRE-FLIGHT CHECK
# =============================================================================
if not RESULTS_PATH.exists() or not CLUBS_PATH.exists():
    raise FileNotFoundError("Master processed data not found. Please run consolidation script first.")

# Load the dataframes
df_matches = pd.read_csv(RESULTS_PATH)
df_clubs = pd.read_csv(CLUBS_PATH)

# =============================================================================
# UTILITY FUNCTION: PERSISTENCE (SAVE TO DISK)
# =============================================================================
# Updated to handle 'ClubId'.
# =============================================================================
def pin_to_watchlist(name, category, club_id):
    # Prepare the new entry with ClubId included
    new_entry = pd.DataFrame([{
        'Name': name, 
        'Type': category, 
        'ClubId': int(club_id), # Ensure ID is stored as an integer
        'Date_Pinned': pd.Timestamp.now().strftime('%Y-%m-%d')
    }])
    
    if WATCHLIST_PATH.exists():
        watchlist = pd.read_csv(WATCHLIST_PATH)
        # Check for duplicates based on name
        if name not in watchlist['Name'].values:
            watchlist = pd.concat([watchlist, new_entry], ignore_index=True)
            watchlist.to_csv(WATCHLIST_PATH, index=False)
            print(f"\n[SUCCESS] '{name}' ({category}) [ID: {club_id}] saved.")
        else:
            print(f"\n[INFO] '{name}' is already in your watchlist.")
    else:
        new_entry.to_csv(WATCHLIST_PATH, index=False)
        print(f"\n[SUCCESS] Created new watchlist and saved '{name}' [ID: {club_id}].")

# =============================================================================
# UTILITY FUNCTION: VIEW CURRENT WATCHLIST
# =============================================================================
def display_watchlist():
    if not WATCHLIST_PATH.exists():
        print("\n[!] The watchlist is currently empty.")
        return

    print("\n" + "-"*60)
    print("CURRENT REGIONAL WATCHLIST")
    print("-"*60)
    
    current_list = pd.read_csv(WATCHLIST_PATH)
    # Sorting for organized display
    current_list = current_list.sort_values(by=['Type', 'Name'])
    
    # Define columns to display
    cols = ['Name', 'Type', 'ClubId', 'Date_Pinned']
    print(current_list[cols].to_string(index=False))
    print("-"*60)

# =============================================================================
# MAIN INTERACTIVE SEARCH LOOP
# =============================================================================
while True:
    print("\n" + "="*60)
    print("MISSION CONTROL: REGIONAL WATCHLIST MANAGER")
    print("="*60)
    print("Commands: [L] List Watchlist  |  [EXIT] Quit")
    
    query = input("\nEnter Club name to search: ").strip()
    
    if query.upper() == 'EXIT':
        print("Exiting Regional Watchlist manager.")
        break
        
    if query.upper() == 'L':
        display_watchlist()
        continue
    
    if not query:
        continue
        
    # 1. Fuzzy Search against Master Club List
    club_choices = df_clubs['Name'].unique().tolist()
    best_matches = process.extract(query, club_choices, limit=5)

    print("\n--- Search Results ---")
    for i, (name, score) in enumerate(best_matches):
        print(f"{i}: [{score}% Match] {name}")
    print("S: Search Again")
    print("E: Exit Manager")

    user_choice = input("\nSelect a number, 'S' to search again, or 'E' to exit: ").upper()

    if user_choice == 'E':
        break
    elif user_choice == 'S' or not user_choice.isdigit():
        continue
    
    idx = int(user_choice)
    if idx >= len(best_matches):
        print("Invalid selection.")
        continue
        
    selected_club_name = best_matches[idx][0]
    
    # 2. DATA LOOKUP: Retrieve the ClubId for the selected club
    # We filter the master club dataframe to get the integer ID
    selected_club_id = df_clubs[df_clubs['Name'] == selected_club_name]['ClubId'].iloc[0]
    
    print(f"\nTarget Selected: {selected_club_name} (Club ID: {selected_club_id})")
    
    # 3. Level Selection
    print("-" * 30)
    print("WATCHLIST LEVEL")
    print("C: Pin the entire Club")
    print("T: Pin a specific Team within this club")
    print("A: Abort (Go back)")
    print("-" * 30)
    
    mode = input("Select level [C/T/A]: ").upper()

    if mode == 'A':
        continue
    elif mode == 'C':
        # Pin the club using the retrieved ID
        pin_to_watchlist(selected_club_name, "Club", selected_club_id)
    elif mode == 'T':
        # Search match data for teams belonging to this club
        all_match_teams = pd.concat([df_matches['Team_A_Name'], df_matches['Team_B_Name']]).unique()
        club_teams = sorted([t for t in all_match_teams if str(selected_club_name).lower() in str(t).lower()])
        
        if not club_teams:
            print(f"No specific teams found for '{selected_club_name}'.")
            if input("Pin the Club instead? (y/n): ").lower() == 'y':
                pin_to_watchlist(selected_club_name, "Club", selected_club_id)
        else:
            print(f"\nTeams discovered for {selected_club_name}:")
            for i, team in enumerate(club_teams):
                print(f"{i}: {team}")
            
            t_choice = input("\nSelect team number to pin (or 'A' to abort): ").upper()
            if t_choice != 'A' and t_choice.isdigit():
                t_idx = int(t_choice)
                if t_idx < len(club_teams):
                    # Even for a specific team, we pin the parent's ClubId for relational tracking
                    pin_to_watchlist(club_teams[t_idx], "Team", selected_club_id)


In [ ]:
import pandas as pd
import os
import re
from pathlib import Path
from thefuzz import process

# =============================================================================
# DIRECTORY CONFIGURATION
# =============================================================================
DATA_DIR = Path('../Data/processed')
CLUB_WATCHLIST_PATH = DATA_DIR / 'regional_watchlist_clubs.csv'
TEAM_WATCHLIST_PATH = DATA_DIR / 'regional_watchlist_teams.csv'

RESULTS_PATH = DATA_DIR / 'events/master_match_results.csv'
CLUBS_PATH = DATA_DIR / 'clubs/master_clubs.csv'

if not RESULTS_PATH.exists() or not CLUBS_PATH.exists():
    raise FileNotFoundError("Master processed data not found. Run consolidation first.")

df_matches = pd.read_csv(RESULTS_PATH)
df_clubs = pd.read_csv(CLUBS_PATH)

# =============================================================================
# UTILITY: DATA RETRIEVAL & PERSISTENCE
# =============================================================================
def get_true_team_id(team_name):
    team_a_data = df_matches[df_matches['Team_A_Name'] == team_name]
    if not team_a_data.empty: return int(team_a_data['Team_A_ID'].iloc[0])
    team_b_data = df_matches[df_matches['Team_B_Name'] == team_name]
    if not team_b_data.empty: return int(team_b_data['Team_B_ID'].iloc[0])
    return -1

def pin_club(name, club_id):
    new_entry = pd.DataFrame([{'Name': name, 'ClubId': int(club_id), 'Date_Pinned': pd.Timestamp.now().strftime('%Y-%m-%d')}])
    if CLUB_WATCHLIST_PATH.exists():
        df = pd.read_csv(CLUB_WATCHLIST_PATH)
        if name not in df['Name'].values:
            pd.concat([df, new_entry], ignore_index=True).to_csv(CLUB_WATCHLIST_PATH, index=False)
            print(f"[SUCCESS] Club '{name}' saved.")
        else: print(f"[INFO] '{name}' already exists.")
    else:
        new_entry.to_csv(CLUB_WATCHLIST_PATH, index=False)

def pin_team(name, team_id):
    new_entry = pd.DataFrame([{'Name': name, 'TeamId': int(team_id), 'Date_Pinned': pd.Timestamp.now().strftime('%Y-%m-%d')}])
    if TEAM_WATCHLIST_PATH.exists():
        df = pd.read_csv(TEAM_WATCHLIST_PATH)
        if name not in df['Name'].values:
            pd.concat([df, new_entry], ignore_index=True).to_csv(TEAM_WATCHLIST_PATH, index=False)
            print(f"[SUCCESS] Team '{name}' saved.")
        else: print(f"[INFO] '{name}' already exists.")
    else:
        new_entry.to_csv(TEAM_WATCHLIST_PATH, index=False)

# =============================================================================
# FUNCTION: ADD MULTIPLE TEAMS (SMART SEARCH)
# =============================================================================
def add_multiple_teams():
    if not CLUB_WATCHLIST_PATH.exists():
        print("\n[!] Club watchlist not found.")
        return
        
    clubs_df = pd.read_csv(CLUB_WATCHLIST_PATH)
    if clubs_df.empty: return

    print("\n--- SELECT A PINNED CLUB ---")
    for i, name in enumerate(clubs_df['Name']):
        print(f"{i}: {name}")
        
    c_idx = input("\nEnter the Club number (A to abort): ").upper()
    if c_idx == 'A' or not c_idx.isdigit() or not (0 <= int(c_idx) < len(clubs_df)): return
        
    selected_club = clubs_df.iloc[int(c_idx)]['Name']
    
    # ---------------------------------------------------------
    # SMART KEYWORD LOGIC
    # ---------------------------------------------------------
    # We strip common volleyball suffixes to get a "Core Name"
    # Example: "The St. James Volleyball Club" -> "The St. James"
    suffixes = [
        'Volleyball Club', 'Volleyball Academy', 'Volleyball', 
        'VBC', 'VB', 'Club', 'Academy', 'Juniors'
    ]
    search_keyword = selected_club
    for s in suffixes:
        # Use regex to remove suffix case-insensitively
        search_keyword = re.sub(rf'\b{re.escape(s)}\b', '', search_keyword, flags=re.IGNORECASE).strip()

    # Attempt First Search
    all_teams = pd.concat([df_matches['Team_A_Name'], df_matches['Team_B_Name']]).dropna().unique()
    club_teams = sorted([t for t in all_teams if search_keyword.lower() in str(t).lower()])
    
    # ---------------------------------------------------------
    # MANUAL OVERRIDE LOGIC
    # ---------------------------------------------------------
    if not club_teams:
        print(f"\n[!] No teams found using automatic keyword: '{search_keyword}'")
        search_keyword = input(f"Enter a custom keyword to find teams for '{selected_club}' (A to abort): ")
        if search_keyword.upper() == 'A': return
        club_teams = sorted([t for t in all_teams if search_keyword.lower() in str(t).lower()])

    if not club_teams:
        print(f"Still no teams found for '{search_keyword}'.")
        return
        
    print(f"\n--- TEAMS FOUND FOR '{search_keyword}' ---")
    for i, team in enumerate(club_teams):
        print(f"{i}: {team}")
        
    print("\nEnter numbers separated by commas (e.g., 0, 2), type 'ALL', or 'A' to abort.")
    selections = input("Selections: ").upper()
    
    if selections == 'A': return
    
    indices = range(len(club_teams)) if selections == 'ALL' else \
              [int(x.strip()) for x in selections.split(',') if x.strip().isdigit() and 0 <= int(x.strip()) < len(club_teams)]
        
    for i in indices:
        team_name = club_teams[i]
        pin_team(team_name, get_true_team_id(team_name))

# =============================================================================
# MAIN INTERACTIVE LOOP
# =============================================================================
while True:
    print("\n" + "="*65)
    print("MISSION CONTROL: REGIONAL WATCHLIST MANAGER")
    print("Commands: [L] List All | [R] Remove | [M] Add Multiple | [EXIT]")
    print("Search:   [C] Club     | [T] Team")
    print("="*65)
    
    cmd = input("\nSelect Mode or Command: ").strip().upper()
    
    if cmd == 'EXIT': break
    if cmd == 'L':
        for label, path in [("CLUBS", CLUB_WATCHLIST_PATH), ("TEAMS", TEAM_WATCHLIST_PATH)]:
            print(f"\n--- {label} ---")
            if path.exists(): print(pd.read_csv(path).to_string(index=False))
            else: print("[Empty]")
        continue
    if cmd == 'R':
        # (Assuming remove function is as defined in v6)
        continue 
    if cmd == 'M':
        add_multiple_teams()
        continue
    
    if cmd == 'C':
        query = input("Enter Club name: ")
        matches = process.extract(query, df_clubs['Name'].unique().tolist(), limit=5)
        for i, (n, s) in enumerate(matches): print(f"{i}: [{s}%] {n}")
        idx = input("\nSelect number to pin (A to abort): ").upper()
        if idx != 'A' and idx.isdigit():
            name = matches[int(idx)][0]
            cid = df_clubs[df_clubs['Name'] == name]['ClubId'].iloc[0]
            pin_club(name, int(cid))

    elif cmd == 'T':
        query = input("Enter Team name: ")
        teams = pd.concat([df_matches['Team_A_Name'], df_matches['Team_B_Name']]).dropna().unique().tolist()
        matches = process.extract(query, teams, limit=5)
        for i, (n, s) in enumerate(matches): print(f"{i}: [{s}%] {n}")
        idx = input("\nSelect number to pin (A to abort): ").upper()
        if idx != 'A' and idx.isdigit():
            name = matches[int(idx)][0]
            pin_team(name, get_true_team_id(name))
